# 多语言宣传技术检测模型测试

## 功能说明
- 支持三种模型配置切换
- 计算标准多标签F1和文章级别F1
- 阈值优化
- 支持英语(en)、俄语(ru)、波兰语(po)测试

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory of this repository
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 Task 3 development data
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirectories

# CheckThat! 2024 Task 3 data
CHECKTHAT_DIR = "your/path/here"

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## 1. 导入库和基础配置

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import os
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import boto3
import tempfile
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, AutoModel
from collections import defaultdict
import warnings
import json
warnings.filterwarnings('ignore')

# 设置环境变量
os.environ["TOKENIZERS_PARALLELISM"] = 'false'
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("✓ Libraries imported")

## 2. 模型配置选择 ⚙️

**请修改下面的配置来选择要测试的模型：**
- `MODEL_CONFIG`: 选择 1, 2, 或 3
- `TEST_LANGUAGE`: 选择 'en', 'ru', 或 'po'

In [ ]:
# ==================== 模型配置选择 ====================
# 1: Base Model - 纯XLM-RoBERTa-base，无解释增强
# 2: Concatenation - 原始文本与解释通过[SEP]拼接
# 3: Dual-Encoder - 双编码器并行处理，双向交叉注意力融合

MODEL_CONFIG = 2  # 👈 修改这里选择配置 (1, 2, 或 3)

MAX_EXPLANATION_LENGTH = 128  # 解释的最大长度

# ==================== 测试语言选择 ====================
# 可选: 'en' (英语), 'ru' (俄语), 'po' (波兰语)
TEST_LANGUAGE = 'ru'  # 👈 修改这里选择测试语言

# ✅ 修改后的模型配置映射（修正了路径）1
MODEL_CONFIGS = {
    1: {
        'name': 'Base_Model',
        # ✅ 修正：去掉了 explanation_enhanced/ 路径
        's3_key': 'multi_label_models/xlm-roberta-base_10_bce_all_train_multi_label.ckpt',
        'description': '纯XLM-RoBERTa-base，无解释增强',
        'use_explanations': False,
        'model_size': 'base',
        'max_length': 128  # ✅ 新增
    },
    2: {
        'name': 'Concatenation',
        's3_key': 'multi_label_models/explanation_enhanced/xlm-roberta-base_10_bce_explanation_enhanced.ckpt',
        'description': '原始文本与解释通过[SEP]拼接',
        'use_explanations': True,
        'model_size': 'base',
        'max_length': 256  # ✅ 新增
    },
    3: {
        'name': 'Dual_Encoder',
        's3_key': 'multi_label_models/explanation_enhanced/xlm-roberta-base_10_bce_2enew_explanation_enhanced.ckpt',
        'description': '双编码器并行处理，双向交叉注意力融合',
        'use_explanations': True,
        'model_size': 'base',
        'max_length': 256  # ✅ 新增
    }
}

# 语言配置映射（保持不变）
LANGUAGE_CONFIGS = {
    'en': {'name': 'English', 'label': '英语'},
    'ru': {'name': 'Russian', 'label': '俄语'},
    'po': {'name': 'Polish', 'label': '波兰语'}
}

# 选择配置
selected_config = MODEL_CONFIGS[MODEL_CONFIG]
language_config = LANGUAGE_CONFIGS[TEST_LANGUAGE]

# ✅ 新增：根据配置动态设置模型和最大长度
model_name = f"xlm-roberta-{selected_config['model_size']}"
MAX_LENGTH = selected_config['max_length']

print(f"\n{'='*60}")
print(f"选择的模型: {selected_config['name']}")
print(f"描述: {selected_config['description']}")
print(f"使用解释: {selected_config['use_explanations']}")
print(f"模型规模: {selected_config['model_size']}")
print(f"最大长度: {MAX_LENGTH}")
print(f"\n测试语言: {language_config['name']} ({language_config['label']})")
print(f"{'='*60}")

## 3. 参数配置

In [ ]:
# ==================== S3和模型参数配置 ====================
ENDPOINT = "https://your-s3-endpoint"  # Set to your S3-compatible endpoint
S3_BUCKET = "your-s3-bucket"
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

# Model hyperparameters
model_name = 'xlm-roberta-base'
MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5

# 解释数据文件路径（如果使用解释增强）
EXPLANATIONS_FILE = "your/path/here"  # Update to your local path

# Output directory
OUTPUT_DIR = 'your/path/here'  # Update to your local path
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n使用设备: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"可用显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

print(f"\n✓ 输出目录: {OUTPUT_DIR}")

## 4. 辅助函数定义

In [ ]:
def download_from_s3(bucket_name, s3_key, local_file_path):
    """从S3下载文件"""
    try:
        s3 = boto3.resource(
            's3',
            endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY
        )
        
        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
        
        print(f"正在从 S3 下载文件...")
        s3.Bucket(bucket_name).download_file(s3_key, local_file_path)
        print(f"✓ 成功下载文件到 {local_file_path}")
        return True
    except Exception as e:
        print(f"✗ 从S3下载失败: {str(e)}")
        return False

print("✓ S3 download function defined")

In [ ]:
def load_label_classes_and_create_mapping(language_code='en'):
    """加载标签并创建智能映射"""
    from collections import Counter
    
    # 直接使用文件中的所有25个标签
    classes_file = "your/path/here"  # Update to your local path
    with open(classes_file, "r", encoding='utf-8') as f:
        label_classes = [line.strip() for line in f.readlines() if line.strip()]
    
    print(f"✓ 加载了 {len(label_classes)} 个标签")
    
    # 创建1对1映射
    label_mapping = {label: label for label in label_classes}
    
    # 从数据中统计频率（用于参考）
    labels_file = f'your/path/here'  # SemEval 2023 data directory
    label_frequency = Counter()
    
    if os.path.exists(labels_file):
        with open(labels_file, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 3 and parts[2].strip():
                    for label in parts[2].split(','):
                        label = label.strip()
                        if label in label_mapping:
                            label_frequency[label] += 1

    # if hasattr(model, 'config') and hasattr(model.config, 'id2label'):
    #     # 方法1: 从模型配置获取
    #     label_classes = [model.config.id2label[i] for i in range(len(model.config.id2label))]
    #     print(f"✓ 从模型获取到 {len(label_classes)} 个标签")
    # else:
    #     # 方法2: 手动补全（如果模型没有配置）
    #     label_classes = [
    #         'Name_Calling-Labeling',
    #         'Guilt_by_Association', 
    #         'Doubt',
    #         'Appeal_to_Hypocrisy',
    #         'Questioning_the_Reputation',
    #         'Flag_Waving',
    #         'Appeal_to_Authority',
    #         'Appeal_to_Popularity',
    #         'Appeal_to_Values',
    #         'Appeal_to_Fear-Prejudice',
    #         'Straw_Man',
    #         'Red_Herring',
    #         'Whataboutism',
    #         'Appeal_to_Pity',
    #         'Causal_Oversimplification',
    #         'False_Dilemma-No_Choice',
    #         'Consequential_Oversimplification',
    #         'False_Equivalence',
    #         'Slogans',
    #         'Conversation_Killer',
    #         'Appeal_to_Time',
    #         'Loaded_Language',
    #         'Obfuscation-Vagueness-Confusion',
    #         'Exaggeration-Minimisation',  # ⭐ 缺失的标签
    #         'Repetition'  # ⭐ 缺失的标签
    #     ]
    #     print(f"✓ 使用手动定义的 {len(label_classes)} 个标签")
    #     label_mapping = {k: v for k, v in label_mapping.items() if v in label_classes}

    # ✅ 删除了对model的依赖,直接使用手动定义的标签
# 或者从文件加载的标签已经足够了

# 如果前面已经从文件加载了标签,就不需要重新定义
# 如果文件不存在或标签不完整,使用手动定义的25个标签

    if len(label_classes) < 25:  # 只有在标签不足时才补充
        label_classes = [
            'Name_Calling-Labeling',
            'Guilt_by_Association', 
            'Doubt',
            'Appeal_to_Hypocrisy',
            'Questioning_the_Reputation',
            'Flag_Waving',
            'Appeal_to_Authority',
            'Appeal_to_Popularity',
            'Appeal_to_Values',
            'Appeal_to_Fear-Prejudice',
            'Straw_Man',
            'Red_Herring',
            'Whataboutism',
            'Appeal_to_Pity',
            'Causal_Oversimplification',
            'False_Dilemma-No_Choice',
            'Consequential_Oversimplification',
            'False_Equivalence',
            'Slogans',
            'Conversation_Killer',
            'Appeal_to_Time',
            'Loaded_Language',
            'Obfuscation-Vagueness-Confusion',
            'Exaggeration-Minimisation',
            'Repetition'
        ]
        print(f"✓ 使用手动定义的 {len(label_classes)} 个标签")



    print(f"✓ 截断到模型支持的 {len(label_classes)} 个标签")
    return label_classes, label_mapping, label_frequency


In [ ]:
def load_language_data(data_folder, labels_file, label_mapping):  # ✅ 添加参数
    """
    加载特定语言的测试数据
    
    Args:
        data_folder: 文章文本文件夹路径
        labels_file: 标签文件路径
    
    Returns:
        DataFrame包含 id, line, text, labels
    """
    # 读取文本数据
    text_data = []
    skipped_files = 0
    
    print(f"读取文本文件: {data_folder}")
    for filename in tqdm(filter(lambda x: x.endswith('.txt'), os.listdir(data_folder))):
        article_id = filename.split('.')[0]
        try:
            with open(os.path.join(data_folder, filename), 'r', encoding='utf-8') as f:
                content = f.read()
            lines = list(enumerate(content.splitlines(), 1))
            text_data.extend([(article_id, line_num, line_text) for line_num, line_text in lines])
        except UnicodeDecodeError:
            skipped_files += 1
            print(f"跳过无法解码的文件: {filename}")
    
    if skipped_files > 0:
        print(f"总共跳过 {skipped_files} 个文件")
    
    # 创建文本DataFrame
    df_text = pd.DataFrame(text_data, columns=['id', 'line', 'text'])
    df_text['line'] = df_text['line'].astype(int)
    df_text = df_text[df_text['text'].str.strip().str.len() > 0].copy()
    df_text = df_text.set_index(['id', 'line'])
    
    # 读取标签数据
    labels_data = []
    if os.path.exists(labels_file):
        with open(labels_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t')
                if len(parts) >= 3 and parts[2].strip(): 
                    article_id = 'article' + parts[0] 
                    line_num = int(parts[1])
                    label_combination = parts[2].strip()
                    
                    # 🔧 修复：分割多标签并应用映射
                    individual_labels = label_combination.split(',')
                    mapped_labels = []
                    
                    for label in individual_labels:
                        label = label.strip()  # ← 也加上 .strip()
                        if label in label_mapping:
                            mapped_label = label_mapping[label]
                            mapped_labels.append(mapped_label)
                    if mapped_labels:
                        labels_data.append([article_id, line_num, ','.join(mapped_labels)])
        
        df_labels = pd.DataFrame(labels_data, columns=['id', 'line', 'label'])
        df_labels['line'] = df_labels['line'].astype(int)
        
        # 聚合同一行的多个标签
        df_labels_agg = df_labels.groupby(['id', 'line'])['label'].apply(lambda x: ','.join(x)).reset_index()
        df_labels_agg.columns = ['id', 'line', 'labels']
        df_labels_agg = df_labels_agg.set_index(['id', 'line'])
        
        # 合并文本和标签
        df = df_text.join(df_labels_agg, how='left')[['text', 'labels']]
        df['labels'] = df['labels'].fillna('')
    else:
        print(f"警告: 标签文件不存在: {labels_file}")
        df = df_text.copy()
        df['labels'] = ''
        df = df[['text', 'labels']]
    
    # 文本清洗
    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # Filter overly long texts
    max_text_len = 1000
    original_len = len(df)
    df = df[df['text'].str.len() <= max_text_len].copy()
    filtered_count = original_len - len(df)
    if filtered_count > 0:
        print(f"过滤了 {filtered_count} 条超长文本 (>{max_text_len} 字符)")
    
    return df.reset_index()

def convert_to_multilabel(df, label_classes):
    """
    将DataFrame转换为多标签格式
    
    Returns:
        ids, lines, texts, multi_label_tensor
    """
    num_labels = len(label_classes)
    
    ids = df['id'].to_numpy()
    lines = df['line'].to_numpy()
    texts = df['text'].to_numpy()
    
    # Build multi-label matrix
    multi_labels = []
    for label_str in df['labels'].fillna('').values:
        label_vec = np.zeros(num_labels, dtype=np.float32)
        if label_str:
            labels = label_str.split(',')
            for label in labels:
                if label in label_classes:
                    label_idx = label_classes.index(label)
                    label_vec[label_idx] = 1.0
        multi_labels.append(label_vec)
    
    multi_labels_tensor = torch.tensor(np.array(multi_labels))
    
    return ids, lines, texts, multi_labels_tensor

def load_explanations_data(explanations_file):
    """加载解释数据"""
    explanations = {}
    try:
        df = pd.read_csv(explanations_file, sep='\t')
        
        # ✅ 关键！必须添加这一行
        df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
        
        for _, row in df.iterrows():
            key = (row['id'], row['text'])
            explanations[key] = row['analysis']
        
        print(f"✅ 成功加载 {len(explanations)} 条解释数据")
    except Exception as e:
        print(f"❌ 加载失败: {str(e)}")
    
    return explanations

print("✓ Data loading functions defined")

## 5. 模型类定义

In [ ]:
import torch.nn as nn
import pytorch_lightning as pl
from transformers import get_linear_schedule_with_warmup

# Base Model / Concatenation 模型类
class MultiLabelClassifierWithExplanations(pl.LightningModule):
    def __init__(self, plm, num_labels, class_weights=None, learning_rate=2e-5, warmup_steps=1000, **kwargs):
        super().__init__()
        self.save_hyperparameters(ignore=['plm', 'class_weights'])
        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        
        if isinstance(plm, str):
            config = AutoConfig.from_pretrained(plm, num_labels=num_labels, problem_type="multi_label_classification")
            self.plm = AutoModelForSequenceClassification.from_pretrained(plm, config=config)
        else:
            self.plm = plm
        
        if hasattr(self.plm, "gradient_checkpointing_enable"):
            self.plm.gradient_checkpointing_enable()
        
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    
    def forward(self, samples, masks):
        x = self.plm(samples, masks)
        return torch.clamp(x.logits, min=-10.0, max=10.0)
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=0.001)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=self.warmup_steps, num_training_steps=10000)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step", "frequency": 1}}

# Dual-Encoder 模型类
class DualEncoderMultiLabelClassifier(pl.LightningModule):
    def __init__(self, plm, num_labels, class_weights=None, learning_rate=1e-5, warmup_steps=1000, **kwargs):
        super().__init__()
        self.save_hyperparameters(ignore=['plm', 'class_weights'])
        
        if isinstance(plm, str):
            self.text_encoder = AutoModel.from_pretrained(plm)
            self.explanation_encoder = AutoModel.from_pretrained(plm)
        else:
            raise ValueError("Dual-Encoder需要plm为字符串")
        
        if hasattr(self.text_encoder, "gradient_checkpointing_enable"):
            self.text_encoder.gradient_checkpointing_enable()
            self.explanation_encoder.gradient_checkpointing_enable()
        
        self.hidden_size = self.text_encoder.config.hidden_size
        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        
        self.text_to_exp_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.exp_to_text_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        self.fusion_layer = nn.Sequential(
            nn.Linear(self.hidden_size * 4, self.hidden_size * 2), nn.LayerNorm(self.hidden_size * 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(self.hidden_size * 2, self.hidden_size), nn.LayerNorm(self.hidden_size), nn.ReLU(), nn.Dropout(0.1)
        )
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    
    def forward(self, text_ids, text_mask, exp_ids=None, exp_mask=None):
        text_outputs = self.text_encoder(input_ids=text_ids, attention_mask=text_mask, return_dict=True)
        text_hidden = text_outputs.last_hidden_state
        text_cls = text_hidden[:, 0, :]
        
        if exp_ids is None or exp_mask is None:
            return self.classifier(text_cls)
        
        exp_outputs = self.explanation_encoder(input_ids=exp_ids, attention_mask=exp_mask, return_dict=True)
        exp_hidden = exp_outputs.last_hidden_state
        exp_cls = exp_hidden[:, 0, :]
        
        text_to_exp_out, _ = self.text_to_exp_attention(query=text_hidden, key=exp_hidden, value=exp_hidden, key_padding_mask=~exp_mask.bool())
        exp_to_text_out, _ = self.exp_to_text_attention(query=exp_hidden, key=text_hidden, value=text_hidden, key_padding_mask=~text_mask.bool())
        
        combined = torch.cat([text_cls, exp_cls, text_to_exp_out[:, 0, :], exp_to_text_out[:, 0, :]], dim=1)
        fused = self.fusion_layer(combined)
        return self.classifier(fused)
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=self.warmup_steps, num_training_steps=10000)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step", "frequency": 1}}

print("✓ Model classes defined")

## 6. 模型加载

In [ ]:
def load_model_from_s3(s3_key, num_labels):
    """使用PyTorch Lightning的load_from_checkpoint方法加载模型"""
    local_checkpoint_path = os.path.join(tempfile.gettempdir(), "model_checkpoint.ckpt")
    
    print(f"正在下载模型...")
    if not download_from_s3(S3_BUCKET, s3_key, local_checkpoint_path):
        print("✗ 下载失败")
        return None
    
    print(f"✓ 下载完成")
    
    # 根据配置选择模型类
    use_dual_encoder = (MODEL_CONFIG == 3)
    ModelClass = DualEncoderMultiLabelClassifier if use_dual_encoder else MultiLabelClassifierWithExplanations
    
    try:
        print(f"正在加载模型（{ModelClass.__name__}）...")
        model = ModelClass.load_from_checkpoint(
            local_checkpoint_path,
            plm=model_name,
            num_labels=num_labels,
            strict=False
        )
        
        total_params = sum(p.numel() for p in model.parameters())
        print(f"✓ 加载成功！参数量: {total_params:,}")
        
        if total_params < 20_000_000:
            print(f"⚠️  参数量异常！")
        
    except Exception as e:
        print(f"✗ 加载失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
    
    model.eval()
    model = model.to(device)
    print(f"✓ 模型已移至 {device}")
    return model

print("✓ Model loading function defined")

In [ ]:
# 加载分词器
print("加载分词器...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
print("✓ 分词器加载完成")

In [ ]:
# ==================== 加载标签和创建映射 ====================
print("\n" + "="*80)
print("加载标签和创建映射")
print("="*80)

label_classes, label_mapping, label_frequency = load_label_classes_and_create_mapping(TEST_LANGUAGE)

print(f"✓ 标签类别数: {len(label_classes)}")
print(f"✓ 标签映射数: {len(label_mapping)}")
print()

# ==================== 构建数据路径 ====================
# Build data paths
data_folder = f'your/path/here'  # SemEval 2023 data directory
labels_file = f'your/path/here'  # SemEval 2023 data directory

print(f"\n{'='*80}")
print(f"测试数据集: {language_config['name']} ({language_config['label']})")
print('='*80)

# Verify paths
if not os.path.exists(data_folder):
    print(f"✗ 数据文件夹不存在: {data_folder}")
    raise FileNotFoundError(f"找不到数据文件夹")

if not os.path.exists(labels_file):
    print(f"警告: 标签文件不存在: {labels_file}")
    print("将继续测试，但无法计算准确率")

# Load data
print(f"\n加载 {language_config['name']} 数据...")
df = load_language_data(data_folder, labels_file, label_mapping)  # 现在可以用了！
print(f"✓ 加载了 {len(df)} 条数据")

# 显示数据样本
print(f"\n数据样本（前3条）:")
print(df[['id', 'text', 'labels']].head(3))

In [ ]:
# Load model
print(f"\n加载模型: {selected_config['name']}")
model = load_model_from_s3(selected_config['s3_key'], len(label_classes))

if model is None:
    print("✗ 模型加载失败，请检查配置")
else:
    print("\n=== 模型验证 ===")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"总参数量: {total_params:,}")
    
    # 测试前向传播
    test_input = tokenizer(["Test"], padding="max_length", truncation=True, 
                          max_length=MAX_LENGTH, return_tensors="pt").to(device)
    
    with torch.no_grad():
        if isinstance(model, DualEncoderMultiLabelClassifier):
            outputs = model(test_input["input_ids"], test_input["attention_mask"])
        else:
            outputs = model(test_input["input_ids"], test_input["attention_mask"])
        
        probs = torch.sigmoid(outputs)
        print(f"✓ 前向传播成功")
        print(f"  输出形状: {outputs.shape}")
        print(f"  概率范围: [{probs.min():.4f}, {probs.max():.4f}]")

## 7. 加载解释数据（如果需要）

In [ ]:
explanations = None
if selected_config['use_explanations']:
    print("\n尝试加载解释数据...")
    explanations = load_explanations_data(EXPLANATIONS_FILE)
else:
    print("\n当前配置不使用解释数据")

## 8. 评估函数定义

In [ ]:
def optimize_thresholds(model, tokenizer, test_df, labels_name, explanations=None):
    """
    为每个标签优化阈值（基于预测概率分布的无监督方式）
    
    Args:
        model: 训练好的模型
        tokenizer: 分词器
        test_df: 测试数据DataFrame
        labels_name: 标签名称列表
        explanations: 解释数据字典（可选）
    
    Returns:
        optimized_thresholds: 每个标签的最优阈值字典
        all_probabilities: 所有样本的预测概率矩阵
    """
    print("\n" + "="*70)
    print("优化标签阈值（基于预测概率分布）")
    print("="*70)
    
    model.eval()
    all_probabilities = []
    
    # 第一遍：获取所有样本的预测概率
    print("正在收集预测概率...")
    # for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="获取概率"):
    #     batch = test_df.iloc[i:i+BATCH_SIZE]
    #     batch_texts = batch['text'].tolist()
        
    #     # 准备解释数据
    #     batch_exps = []
    #     for _, row in batch.iterrows():
    #         key = (row['id'], row['text'])
    #         exp = explanations.get(key, "") if explanations else ""
    #         batch_exps.append(exp if exp else "")

    for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="获取概率"):
        batch = test_df.iloc[i:i+BATCH_SIZE]
        
        # ✅✅✅ 新增：根据配置准备输入 ✅✅✅
        if MODEL_CONFIG == 1:
            # Config 1: Base Model - 只用原文
            batch_texts = batch['text'].tolist()
            
        elif MODEL_CONFIG == 2:
            # Config 2: Concatenation - 文本 + [SEP] + 解释
            batch_texts = []
            for _, row in batch.iterrows():
                text = row['text']
                key = (row['id'], text)
                
                if explanations and key in explanations:
                    combined = f"{text} [SEP] {explanations[key]}"
                    batch_texts.append(combined)
                else:
                    batch_texts.append(text)
        
        else:
            # Config 3 或其他配置
            batch_texts = batch['text'].tolist()
        # ✅✅✅ 新增结束 ✅✅✅    
        
        try:
            # 对文本进行编码（统一处理）
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(model.device)
            
            with torch.no_grad():
                # 自动检测模型类型
                if hasattr(model, 'text_encoder'):
                    # 配置3: Dual-Encoder（需要解释但现在传None）
                    logits = model(
                        text_ids=inputs["input_ids"],
                        text_mask=inputs["attention_mask"],
                        exp_ids=None,
                        exp_mask=None
                    )
                else:
                    # 配置1和2: Base Model / Concatenation
                    logits = model(inputs["input_ids"], inputs["attention_mask"])
                
                probabilities = torch.sigmoid(logits)
                all_probabilities.append(probabilities.cpu().numpy())
                
        except Exception as e:
            print(f"批次 {i//batch_size} 处理出错: {str(e)}")
            empty_probs = np.zeros((len(batch_texts), len(labels_name)))
            all_probabilities.append(empty_probs)
    
    # 合并所有概率
    all_probabilities = np.concatenate(all_probabilities, axis=0)
    
    print(f"✓ 概率收集完成")
    print(f"  形状: {all_probabilities.shape}")
    print(f"  范围: [{all_probabilities.min():.4f}, {all_probabilities.max():.4f}]")
    
    # 为每个标签基于概率分布优化阈值
    # 在函数的末尾，找到优化阈值的循环

    num_model_outputs = all_probabilities.shape[1]
    if len(labels_name) > num_model_outputs:
        print(f"⚠️ 截断 label_classes: {len(labels_name)} → {num_model_outputs}")
        labels_name = labels_name[:num_model_outputs]
        
    optimized_thresholds = {}

    for i, label in enumerate(labels_name):
        label_probs = all_probabilities[:, i]
        
        # 分析概率分布
        prob_75 = np.percentile(label_probs, 75)
        prob_90 = np.percentile(label_probs, 90)
        
        # 🔧 修复：使用基于分布的策略
        if prob_90 < 0.3:
            threshold = max(0.25, prob_75)
        elif prob_90 < 0.5:
            threshold = max(0.3, prob_75)
        else:
            threshold = max(0.35, min(0.55, (prob_75 + prob_90) / 2))
        
        optimized_thresholds[label] = threshold
        
        predicted_count = (label_probs > threshold).sum()
        print(f"  {label:40s}: 阈值={threshold:.2f} | 预测数={predicted_count}")
    
    print("\n阈值统计:")
    threshold_values = list(optimized_thresholds.values())
    print(f"  最小阈值: {min(threshold_values):.2f}")
    print(f"  最大阈值: {max(threshold_values):.2f}")
    print(f"  平均阈值: {np.mean(threshold_values):.2f}")
    
    return optimized_thresholds, all_probabilities


print("✓ Threshold optimisation function updated")

In [ ]:
def calculate_multilabel_f1(ids, texts, true_labels, predictions, label_classes):
    """计算多标签分类F1（文章级别）"""
    print(f"\n{'='*70}")
    print("多标签分类 F1 (Article-Level Technique Detection)")
    print('='*70)
    
    # 转换为spans格式
    true_spans = []
    pred_spans = []
    
    for i in range(len(ids)):
        article_id = ids[i]
        
        # 真实标签
        true_label_indices = np.where(true_labels[i] == 1)[0]
        for idx in true_label_indices:
            technique = label_classes[idx]
            true_spans.append((article_id, technique, 0, 0))
        
        # Run inference标签
        pred_label_indices = np.where(predictions[i] == 1)[0]
        for idx in pred_label_indices:
            technique = label_classes[idx]
            pred_spans.append((article_id, technique, 0, 0))
    
    # 按文章分组
    true_by_article = defaultdict(set)
    pred_by_article = defaultdict(set)
    
    for article_id, technique, _, _ in true_spans:
        true_by_article[article_id].add(technique)
    
    for article_id, technique, _, _ in pred_spans:
        pred_by_article[article_id].add(technique)
    
    all_articles = sorted(set(true_by_article.keys()) | set(pred_by_article.keys()))
    all_techniques = set()
    for techs in true_by_article.values():
        all_techniques.update(techs)
    for techs in pred_by_article.values():
        all_techniques.update(techs)
    all_techniques = sorted(list(all_techniques))
    
    # 创建二值矩阵
    y_true = []
    y_pred = []
    
    for article_id in all_articles:
        true_techs = true_by_article.get(article_id, set())
        pred_techs = pred_by_article.get(article_id, set())
        
        true_vec = [1 if tech in true_techs else 0 for tech in all_techniques]
        pred_vec = [1 if tech in pred_techs else 0 for tech in all_techniques]
        
        y_true.append(true_vec)
        y_pred.append(pred_vec)
    
    # 计算整体指标
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # 计算每个技术的指标
    tech_metrics = []
    for i, tech in enumerate(all_techniques):
        true_col = [row[i] for row in y_true]
        pred_col = [row[i] for row in y_pred]
        
        p = precision_score(true_col, pred_col, zero_division=0)
        r = recall_score(true_col, pred_col, zero_division=0)
        f1 = f1_score(true_col, pred_col, zero_division=0)
        support = sum(true_col)
        
        tech_metrics.append({
            'technique': tech,
            'precision': p,
            'recall': r,
            'f1': f1,
            'support': support
        })
    
    tech_df = pd.DataFrame(tech_metrics)
    tech_df = tech_df.sort_values('f1', ascending=False)
    
    # 打印结果
    print(f"\n文章数: {len(all_articles)} | 技术类型数: {len(all_techniques)}")
    print(f"\nMacro F1:    {macro_f1:.4f} ({macro_f1*100:.2f}%)")
    print(f"Micro F1:    {micro_f1:.4f} ({micro_f1*100:.2f}%)")
    print(f"Weighted F1: {weighted_f1:.4f} ({weighted_f1*100:.2f}%)")
    
    print(f"\n各技术类型详细指标:")
    print(tech_df.to_string(index=False))
    
    results = {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'weighted_f1': weighted_f1,
        'tech_metrics': tech_df,
        'all_techniques': all_techniques,
        'num_articles': len(all_articles)
    }
    
    return results

print("✓ Multi-label F1 function defined")

## 9. 加载测试数据

In [ ]:
# ==================== 加载标签和创建映射 ====================
print("\n" + "="*80)
print("加载标签和创建映射")
print("="*80)

label_classes, label_mapping, label_frequency = load_label_classes_and_create_mapping(TEST_LANGUAGE)

print(f"✓ 标签类别数: {len(label_classes)}")
print(f"✓ 标签映射数: {len(label_mapping)}")
print()

# ==================== 构建数据路径 ====================
# Build data paths
data_folder = f'your/path/here'  # SemEval 2023 data directory
labels_file = f'your/path/here'  # SemEval 2023 data directory

print(f"\n{'='*80}")
print(f"测试数据集: {language_config['name']} ({language_config['label']})")
print('='*80)

# Verify paths
if not os.path.exists(data_folder):
    print(f"✗ 数据文件夹不存在: {data_folder}")
    raise FileNotFoundError(f"找不到数据文件夹")

if not os.path.exists(labels_file):
    print(f"警告: 标签文件不存在: {labels_file}")
    print("将继续测试，但无法计算准确率")

# Load data
print(f"\n加载 {language_config['name']} 数据...")
df = load_language_data(data_folder, labels_file, label_mapping)  # 现在可以用了！
print(f"✓ 加载了 {len(df)} 条数据")

# 显示数据样本
print(f"\n数据样本（前3条）:")
print(df[['id', 'text', 'labels']].head(3))

In [ ]:
# Build data paths
data_folder = f'your/path/here'  # SemEval 2023 data directory
labels_file = f'your/path/here'  # SemEval 2023 data directory

print(f"\n{'='*80}")
print(f"测试数据集: {language_config['name']} ({language_config['label']})")
print('='*80)

# Verify paths
if not os.path.exists(data_folder):
    print(f"✗ 数据文件夹不存在: {data_folder}")
    raise FileNotFoundError(f"找不到数据文件夹")

if not os.path.exists(labels_file):
    print(f"警告: 标签文件不存在: {labels_file}")
    print("将继续测试，但无法计算准确率")

# Load data
print(f"\n加载 {language_config['name']} 数据...")
df = load_language_data(data_folder, labels_file, label_mapping)
print(f"✓ 加载了 {len(df)} 条数据")

# 显示数据样本
print(f"\n数据样本（前3条）:")
print(df[['id', 'text', 'labels']].head(3))

In [ ]:
# 转换为多标签格式
ids, lines, texts, multi_labels = convert_to_multilabel(df, label_classes)

print(f"\n数据统计:")
print(f"  样本数: {len(ids)}")
print(f"  唯一文章数: {len(set(ids))}")
print(f"  标签矩阵形状: {multi_labels.shape}")
print(f"  平均每个样本的标签数: {multi_labels.sum(axis=1).mean():.2f}")

In [ ]:
# 手动测试标签映射
labels_file = f'your/path/here'  # SemEval 2023 data directory

with open(labels_file, 'r') as f:
    lines = f.readlines()

# 提取文件中的所有标签
file_labels = set()
total_lines = 0
lines_with_labels = 0

for line in lines:
    parts = line.strip().split('\t')
    if len(parts) >= 3 and parts[2].strip():
        total_lines += 1
        lines_with_labels += 1
        label_combination = parts[2].strip()
        for label in label_combination.split(','):
            file_labels.add(label.strip())

print(f"标签文件统计:")
print(f"  有标签的行: {lines_with_labels}")
print(f"  唯一标签数: {len(file_labels)}")
print(f"\n文件中的标签:")
for label in sorted(file_labels):
    print(f"  • {label}")

# 检查映射
print(f"\nlabel_mapping 内容:")
print(f"  映射数量: {len(label_mapping)}")
for k, v in list(label_mapping.items())[:5]:
    print(f"  {k} → {v}")

# 检查匹配
matched = file_labels & set(label_mapping.keys())
not_in_mapping = file_labels - set(label_mapping.keys())

print(f"\n匹配分析:")
print(f"  文件中的标签: {len(file_labels)} 个")
print(f"  在映射中的: {len(matched)} 个")
print(f"  不在映射中的: {len(not_in_mapping)} 个")

if not_in_mapping:
    print(f"\n❌ 不在映射中的标签:")
    for label in sorted(not_in_mapping):
        print(f"  • {label}")

## 10. 运行评估

In [ ]:
# 把 ids, lines, texts 合并成 test_df
optimized_thresholds, all_probabilities = optimize_thresholds(
    model, 
    tokenizer, 
    df,               # ← 改成 df（不是 test_df）
    label_classes,
    explanations=explanations
)

# 生成预测
predictions = np.zeros_like(all_probabilities)
for i, label in enumerate(label_classes):
    threshold = optimized_thresholds[label]
    predictions[:, i] = (all_probabilities[:, i] > threshold).astype(int)

# 计算F1
from sklearn.metrics import f1_score
micro_f1 = f1_score(multi_labels.numpy(), predictions, average='micro')
macro_f1 = f1_score(multi_labels.numpy(), predictions, average='macro')

print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

## 11. 计算评估指标

In [ ]:
# 计算标准指标
print(f"\n{'='*70}")
print("标准多标签评估指标")
print('='*70)

metrics = {
    'micro_f1': f1_score(multi_labels, predictions, average='micro', zero_division=0),
    'macro_f1': f1_score(multi_labels, predictions, average='macro', zero_division=0),
    'weighted_f1': f1_score(multi_labels, predictions, average='weighted', zero_division=0),
    'precision': precision_score(multi_labels, predictions, average='micro', zero_division=0),
    'recall': recall_score(multi_labels, predictions, average='micro', zero_division=0)
}

print(f"\n{language_config['name']} 标准评估指标:")
print(f"  Micro F1:     {metrics['micro_f1']:.4f} ({metrics['micro_f1']*100:.2f}%)")
print(f"  Macro F1:     {metrics['macro_f1']:.4f} ({metrics['macro_f1']*100:.2f}%)")
print(f"  Weighted F1:  {metrics['weighted_f1']:.4f} ({metrics['weighted_f1']*100:.2f}%)")
print(f"  Precision:    {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)")
print(f"  Recall:       {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)")

In [ ]:
# 计算多标签F1（文章级别）
multilabel_results = calculate_multilabel_f1(
    ids, texts, multi_labels.numpy(), predictions, label_classes
)